# 02 Data Preparation
I fix the issues from notebook 01 in code, add new features and save a clean file.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/hotel_bookings.csv")
df = raw.copy()
log = []
def note(step, before, after):
    log.append({"step": step, "rows before": before, "rows after": after, "rows removed": before - after})

## 1. Missing values

In [2]:
df["children"] = df["children"].fillna(0).astype(int)
df["country"] = df["country"].fillna("Unknown")
df["has_agent"] = df["agent"].notna().astype(int)        # 86% of bookings came through an agent
df["has_company"] = df["company"].notna().astype(int)    # only 6% booked by a company
df = df.drop(columns=["agent", "company"])               # IDs themselves are not meaningful numbers
print(df.isna().sum().sum(), "missing values left")

0 missing values left


## 2. Categories
`Undefined` meal is the same as `SC` (no meal). Two rows have an `Undefined` market segment.

In [3]:
df["meal"] = df["meal"].replace("Undefined", "SC")
before = len(df)
df = df[df["market_segment"] != "Undefined"]
note("drop market_segment = Undefined", before, len(df))

## 3. Impossible values and outliers

In [4]:
before = len(df)
df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
note("drop bookings with 0 guests", before, len(df))

before = len(df)
df = df[(df["adr"] >= 0) & (df["adr"] < 5000)]
note("drop negative ADR and the single 5400 EUR outlier", before, len(df))

Very long lead times (700 days) and 50+ previous cancellations are rare but possible, so I keep them. Tree models aren't affected much, and for logistic regression I scale the features.

## 4. Duplicates
27% of rows are exact copies. There is no booking ID, so they can be separate rooms in the same group booking (same agent, dates and price). Dropping them would change the cancel rate from 37% to 27%.

I keep them. In notebook 04 I split by time so copies can't be in both train and test, and I also test the model without duplicates.

In [5]:
df["is_duplicate"] = df.duplicated(keep="first").astype(int)
print("Rows flagged as later duplicates:", df["is_duplicate"].sum())

Rows flagged as later duplicates: 31997


## 5. Data types and dates

In [6]:
month_num = {m: i for i, m in enumerate(
    ["January","February","March","April","May","June","July","August","September","October","November","December"], 1)}
df["arrival_month"] = df["arrival_date_month"].map(month_num)
df["arrival_date"] = pd.to_datetime(dict(year=df["arrival_date_year"], month=df["arrival_month"], day=df["arrival_date_day_of_month"]))
df["booking_date"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
df.dtypes.value_counts()

int64             21
str               12
datetime64[us]     2
float64            1
Name: count, dtype: int64

## 6. New features

In [7]:
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["is_family"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["arrival_dayofweek"] = df["arrival_date"].dt.dayofweek          # 0 = Monday
df["is_domestic"] = (df["country"] == "PRT").astype(int)
top_countries = df["country"].value_counts().head(10).index
df["country_group"] = df["country"].where(df["country"].isin(top_countries), "Other")
df["had_previous_cancel"] = (df["previous_cancellations"] > 0).astype(int)
df[["total_nights", "total_guests", "is_family", "arrival_dayofweek", "is_domestic", "country_group"]].head()

,total_nights,total_guests,is_family,arrival_dayofweek,is_domestic,country_group
0,0,2,0,2,1,PRT
1,0,2,0,2,1,PRT
2,1,1,0,2,0,GBR
3,1,1,0,2,0,GBR
4,2,2,0,2,0,GBR


## 7. Leaky columns
Kept in the file for EDA, not used in the models.

In [8]:
LEAKY = ["reservation_status", "reservation_status_date", "assigned_room_type",
         "booking_changes", "required_car_parking_spaces"]
print("Leaky columns that models must NOT use:", LEAKY)

Leaky columns that models must NOT use: ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'required_car_parking_spaces']


## 8. Cleaning log and save

In [9]:
log_df = pd.DataFrame(log)
log_df.loc[len(log_df)] = ["TOTAL", len(raw), len(df), len(raw) - len(df)]
print(log_df.to_string(index=False))
print(f"\nKept {len(df)/len(raw):.2%} of rows. Cancellation rate after cleaning: {df['is_canceled'].mean():.1%}")
log_df.to_csv("../reports/cleaning_log.csv", index=False)

                                             step  rows before  rows after  rows removed
                  drop market_segment = Undefined       119390      119388             2
                      drop bookings with 0 guests       119388      119208           180
drop negative ADR and the single 5400 EUR outlier       119208      119206             2
                                            TOTAL       119390      119206           184

Kept 99.85% of rows. Cancellation rate after cleaning: 37.1%


In [10]:
df.to_csv("../data/hotel_bookings_clean.csv", index=False)
print("Saved data/hotel_bookings_clean.csv", df.shape)

Saved data/hotel_bookings_clean.csv (119206, 43)
